# Phenotype residualization

Input: round 2's ancestry-filtered keep-list (person IDs) and a phenotype list TSV. Output: one `FID IID Y` file per phenotype x transform x covariate-set combination, matching `GRM-pairs/full_grm_bin/prep_pheno.R`'s expected format.

Protocol (order matters, see Kemper et al. 2021 Methods):
1. residualize `phenotype ~ covariates`
2. trim residuals > 5 SD from the mean
3. standardize surviving residuals to mean 0, var 1, within each sex

Covariates: sex-at-birth + age always included. PCs / zip3 (factor) / SES vars (`zip3_ses_map`) are each independently toggleable — every combination gets run, not just one chosen set.

## Setup

**Run this cell first in a freshly restarted kernel.** If a package (e.g. `rlang`) already got auto-loaded from the system library before this cell runs -- which can happen just from kernel startup -- `.libPaths()` can't retroactively swap it for the newer version installed here; R won't unload/replace an already-attached namespace mid-session. Symptom: `namespace 'rlang' 1.1.6 is already loaded, but >= 1.1.7 is required`. Fix is a full kernel restart, not re-running cells in the same session.

In [ ]:
required_pkgs <- c("dplyr", "tidyr", "readr", "stringr", "e1071", "bigrquery", "allofus")
missing_pkgs <- required_pkgs[!sapply(required_pkgs, requireNamespace, quietly = TRUE)]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs, lib = user_lib)

library(dplyr)
library(tidyr)
library(readr)
library(stringr)
library(bigrquery)
library(allofus)
source("../../scripts/local/residualize_lib.R")   # transform, residualize, export -- shared with the fake-data test

con <- aou_connect()   # BigQuery connection to the CDR, used by pull_phenotype() below

## Inputs

- `KEEP_LIST_PATH`: round 2 output — person IDs passing ancestry filtering.
- `PHENO_LIST_PATH`: TSV describing which phenotypes to pull (see schema below).
- `PC_PATH`: round 1's PCA output (`.eigenvec`), or round 1's AoU-projected `.sscore` once that step is done.
- `ZIP3_SES_TABLE`: AoU's `zip3_ses_map` — exact join path (likely via `observation`, not directly on `person`) still needs confirming against the real workbench, flagged below.

Phenotype list TSV schema (`PHENO_LIST_PATH`):

| column | purpose |
|---|---|
| `phenotype_name` | label used in output filenames/diagnostics |
| `source` | `measurement` / `survey` / `condition` -- picks the extraction path |
| `concept_id` | AoU concept ID (comma-separated for multiple) |
| `value_field` | optional -- which field holds the numeric value |
| `notes` | free text, not used programmatically |

In [ ]:
KEEP_LIST_PATH <- "PLACEHOLDER"   # round 2 output, one person_id per line
PHENO_LIST_PATH <- "../../docs/phenotype_list.tsv"  # starter anthropometric/metabolic panel -- see its header for concept_id confirmation status
PC_PATH <- "PLACEHOLDER"          # round 1 .eigenvec (or AoU-projected .sscore)
OUT_DIR <- "PLACEHOLDER"          # where the FID/IID/Y files get written
REFERENCE_DATE <- "PLACEHOLDER"   # e.g. "2024-01-01" -- single fixed date age is computed from for every phenotype, not per-measurement

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

## Read the ID list and phenotype list

`keep_ids`: everyone downstream gets restricted to this set before anything else -- residualization, transforms, and diagnostics are all computed within this cohort only, not the full AoU population.

In [ ]:
keep_ids <- read_lines(KEEP_LIST_PATH)

pheno_list <- read_tsv(PHENO_LIST_PATH, col_types = cols(.default = "c"))
stopifnot(all(c("phenotype_name", "source", "concept_id") %in% names(pheno_list)))
pheno_list

## Pull phenotype + covariate table

`pull_phenotype()` handles `source == "measurement"` via `allofus::aou_sql()` (most recent value per person, sex/age from `person`). `survey`/`condition` sources aren't wired up yet -- add branches when a phenotype needs them.

**Still unresolved, needs the real workbench to pin down:**
- `zip3_ses_map` join path
- exact gender_concept_id set (45878463/45880669 cover Female/Male; anything else falls into `"Other"` -- confirm that's still exhaustive against the current CDR version)

Everything below this cell (transform, covariate-set configs, residualization, export, diagnostics) doesn't depend on these specifics and is fully generic once `pull_phenotype()` returns a table shaped `person_id, phenotype, age, sex_at_birth`.

In [ ]:
pull_phenotype <- function(row, keep_ids) {
  if (row$source != "measurement") {
    stop(sprintf(
      "pull_phenotype(): source '%s' not implemented -- only 'measurement' is wired up so far",
      row$source
    ))
  }

  query <- sprintf("
    WITH demographics AS (
      SELECT
        person_id,
        DATE_DIFF(DATE '%s', DATE(birth_datetime), YEAR) AS age,
        CASE
          WHEN gender_concept_id = 45878463 THEN 'Female'
          WHEN gender_concept_id = 45880669 THEN 'Male'
          ELSE 'Other'
        END AS sex_at_birth
      FROM {CDR}.person
    ),
    measurements AS (
      SELECT
        person_id,
        value_as_number AS phenotype,
        ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY measurement_date DESC) AS rn
      FROM {CDR}.measurement
      WHERE measurement_concept_id IN (%s)
        AND value_as_number IS NOT NULL
    )
    SELECT d.person_id, m.phenotype, d.age, d.sex_at_birth
    FROM demographics d
    INNER JOIN measurements m ON d.person_id = m.person_id
    WHERE m.rn = 1
  ", REFERENCE_DATE, row$concept_id)

  aou_sql(query) %>%
    collect() %>%   # pull out of BigQuery before any local joins/filters -- a
                     # lazy remote tbl can't be left_join()'d against a local one
    mutate(person_id = as.character(person_id)) %>%
    filter(person_id %in% keep_ids)
}

pull_covariates <- function(keep_ids) {
  pcs <- read_table(PC_PATH, col_types = cols(.default = "d", IID = "c")) %>%
    rename(person_id = IID) %>%
    filter(person_id %in% keep_ids)

  zip3 <- tibble(person_id = character(), zip3 = character())        # TODO: real join
  ses <- tibble(person_id = character(), median_income = double())    # TODO: real join

  list(pcs = pcs, zip3 = zip3, ses = ses)
}

## Optional skew-reducing transform

Rank-based inverse-normal transform (`inverse_normal_transform()` in `residualize_lib.R`) -- chosen over log/Box-Cox since it doesn't assume a particular skew direction or require positive values, so it works uniformly across an arbitrary phenotype list without per-phenotype tuning. Both `raw` and `invnorm` variants get carried through the rest of the pipeline; skewness before/after is reported so it's clear whether the transform actually helped for each phenotype, rather than applying it blindly. See `02_phenotype/notebooks/local/test_residualize_fake_data.ipynb` for a worked example on synthetic data showing this diagnostic actually catching a real skew reduction.

## Covariate-set configs

`build_covariate_sets()` (in `residualize_lib.R`) returns a named list of formula RHS vectors. `sex_at_birth` is handled separately (it's the stratification variable for step 3, not a residualization covariate -- see `residualize_phenotype()` in the lib), so it's not in these formulas.

In [ ]:
pc_cols <- paste0("PC", 1:20)   # matches round 1's 20-PC output
covariate_sets <- build_covariate_sets(pc_cols)
names(covariate_sets)

## Main loop

`run_residualization()` (in `residualize_lib.R`) runs the full cross product of {phenotype} x {raw, invnorm} x {covariate-set}, all exported -- no curation, every combination in `pheno_list` x `covariate_sets` gets written out as an `FID IID Y` file matching `GRM-pairs/full_grm_bin/prep_pheno.R`'s expected format.

In [ ]:
result <- run_residualization(
  pheno_list, keep_ids, pull_phenotype, pull_covariates,
  covariate_sets, OUT_DIR
)

## Diagnostics summary

In [ ]:
result$skew_summary_table   # per phenotype: skewness before/after the invnorm transform

In [ ]:
result$combo_summary_table  # per phenotype x variant x covariate-set: N retained, R^2